# Final Project Ciringoni M2
Deteksi objek kecil kerusakan jagung dengan kelas infected.

Notebook ini mengisi seluruh Bagian 1–15 dari starter dosen. Fokus pelaksanaan minggu ini adalah E2, E3, dan E4. Sumber data merupakan hasil UTS dengan split yang sama. Riwayat eksperimen lokal ketua disimpan bersama asal CSV dan ID run aslinya.

**Status 18 November 2026.** E2 native 1280 menjadi kandidat sementara. E1 lokal M1 memakai freeze 10 dan perlu dilengkapi E1 YOLO26s sesuai starter. E3 lokal dari olahan 640 merupakan percobaan pendahuluan. Sel E3 di sini menyiapkan tiling dari sumber 1280. Eksekusi GPU Kaggle, checkpoint input, dan nomor versi Save & Run All perlu tercatat setelah dijalankan pada akun kelompok.

**Mulai di Kaggle.** Tambahkan ZIP hasil UTS sebagai Add Input. Untuk E2–E4, tambahkan ekspor citra asli 1280 dan checkpoint sebagai dataset privat. Isi INPUT_NATIVE dan CHECKPOINTS. Pilih run pada EVALUASI, lalu gunakan LATIH sesuai kebutuhan. Mode awal hanya memeriksa dataset serta menampilkan arsip progres. Setiap hasil baru memiliki status dihitung pada eksekusi ini.

**Sumber ketua** https://github.com/Jordyee/ciringoni-data-mining/tree/e376ec25680cb6204236a527fdacc45d9a2c8e5f

**Notebook sumber di Kaggle** https://www.kaggle.com/code/davejordygerungan/data-mining-mid

Notebook Final Project dibuat sebagai notebook baru. Di Kaggle gunakan Add Input, pilih Notebook, tempel URL sumber di atas lalu tambahkan output versi UTS yang dipakai kelompok. ZIP hasil UTS pada input akan ditemukan otomatis. Periksa nama arsip dan ringkasan split sebelum melanjutkan. Untuk data native1280 dan checkpoint, tambahkan input privat ketua secara terpisah.

**Sumber dosen** FP_DM_deteksi_objek_kecil_FAW.ipynb dan Project.pdf Bagian Final Project. Metrik utama AP_small pada acuan koordinat 640, dengan mAP50_95 sebagai pembanding. Variasi seed masih perlu diuji. Semua LATIH awal False. Uji test diaktifkan setelah konfigurasi final terkunci.

## Aturan main (berlaku untuk seluruh notebook)

1. **Satu faktor per eksperimen.** Setiap eksperimen E1–E7 hanya mengubah satu hal dibanding baseline E0.
2. **Split dari UTS tidak boleh diubah.** Train/val/test sudah dibagi berkelompok (per `source_id` + dHash) untuk mencegah kebocoran.
3. **Pemilihan model memakai split `val`.** Split `test` hanya dipakai **sekali** di Bagian 12 untuk model-model final.
4. **Seed, jumlah epoch, dan protokol evaluasi sama** untuk semua eksperimen, kecuali faktor yang sedang diuji.
5. **Semua angka masuk `eksperimen.csv`** (Bagian 5). Angka yang tidak tercatat di tabel tidak dinilai.
6. Setiap keputusan subjektif (ambang, jumlah epoch, parameter tiling) **ditulis alasannya** di sel markdown.

## 1. Konfigurasi dan lingkungan

**Langkah:**

1. Instal `ultralytics` versi terbaru yang mendukung YOLO26, beserta `pycocotools` dan `sahi`. Cetak versi `ultralytics`, `torch`, dan nama GPU; versi ini wajib dilaporkan.
2. Impor pustaka yang dibutuhkan (`pathlib`, `json`, `time`, `numpy`, `pandas`, `matplotlib`, `cv2`, `ultralytics.YOLO`).
3. Tetapkan konstanta global:
   - `KELOMPOK = "FP_DM_kelompok_XX"` (awalan wajib `FP_DM_`);
   - `SEED = 42` (samakan dengan UTS);
   - `EPOCHS`, `PATIENCE`, `BATCH`, `IMG_SIZE = 640` — tentukan sekali dan tulis alasannya;
   - `CLASS_NAMES = {0: "infected"}` (sama dengan UTS);
   - path: `INPUT_UTS` (folder Kaggle Dataset hasil UTS), `WORK = /kaggle/working`, `TEMP = /kaggle/temp`.
4. Buat **saklar per eksperimen**, misalnya `LATIH = {"E0": True, "E1": False, ...}`. Saat `False`, notebook harus memuat bobot yang sudah dilatih dari input (Bagian 6, langkah 5), bukan melatih ulang.
5. Atur seed untuk `random`, `numpy`, dan `torch`.

In [ ]:
# Persiapan Kaggle atau Colab. Instalasi hanya bila paket analisis belum tersedia.
import importlib.util, os, subprocess, sys
packages = {'yaml': 'PyYAML', 'matplotlib': 'matplotlib', 'pycocotools': 'pycocotools', 'pandas': 'pandas', 'PIL': 'Pillow'}
missing = [pip for mod,pip in packages.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
from pathlib import Path
KELOMPOK = 'FP_DM_Ciringoni'
NOTEBOOK_SUMBER_KAGGLE = 'https://www.kaggle.com/code/davejordygerungan/data-mining-mid'
SEED, EPOCHS, PATIENCE, BATCH, IMG_SIZE = 42, 5, 5, 4, 640
# Lima epoch adalah anggaran pilot yang mengikuti E2 lokal. Konvergensi perlu dievaluasi.
CLASS_NAMES = {0: 'infected'}
INPUT_ROOT = Path(os.environ.get('CIRINGONI_INPUT_ROOT', '/kaggle/input' if Path('/kaggle').exists() else '/content/input'))
WORK = Path(os.environ.get('CIRINGONI_WORK', '/kaggle/working' if Path('/kaggle').exists() else '/content/working'))
TEMP = Path(os.environ.get('CIRINGONI_TEMP', '/kaggle/temp/ciringoni_m2' if Path('/kaggle').exists() else '/content/temp/ciringoni_m2'))
OUT = WORK / KELOMPOK
OUT.mkdir(parents=True, exist_ok=True);TEMP.mkdir(parents=True, exist_ok=True)
INPUT_UTS = os.environ.get('CIRINGONI_INPUT_UTS', '')
# Isi path ZIP/folder hasil ekspor ulang UTS 1280 atau folder berisi 1280/ dan audit_pasangan_e2.json.
INPUT_NATIVE = ''
KONFIRMASI_NATIVE = False  # True hanya setelah memastikan ekspor ulang dari sumber, ID/split sama.
LATIH = {run: False for run in ['E0', 'E1', 'E2_control_native640', 'E2', 'E3', 'E5', 'E6']}
EVALUASI = []  # Contoh ['E2_control_native640', 'E2', 'E3', 'E4'] setelah input siap.
CHECKPOINTS = {}  # Contoh {'E2': '/kaggle/input/datasets/akun/bobot/E2_native_1280/weights/best.pt'}
RESUME = {}  # Contoh {'E2': '/kaggle/input/datasets/akun/run/last.pt'}
UKUR_CPU = False  # Aktifkan untuk 10 warmup + 100 pengulangan, 4 thread, mesin sama.
DEVICE = 0
EKSPOR_TILE = False
JALANKAN_TEST = False
JALANKAN_INT8 = False
MODEL_FINAL = ''  # Isi ID pada HASIL_VAL sesudah kandidat dikunci.
SAHI_CONFIGS = [{'tile':640,'overlap':.2,'standard':False},
                {'tile':640,'overlap':.2,'standard':True},
                {'tile':512,'overlap':.25,'standard':False}]
SAHI_PILIH = [0]  # Paling banyak 3 konfigurasi, hanya validation.
SAHI_MODEL = 'E2'  # Bisa E3 untuk evaluasi E3b setelah bobot tersedia.
LABEL_KELAS_640 = ''  # Label kelas lama yang sudah ditransformasi ke koordinat640.
LABEL_ACUAN_640 = ''  # Label acuan kelompok yang sudah ditransformasi ke koordinat640.
print('Output',OUT,'| LATIH',LATIH,'| EVALUASI',EVALUASI)

In [ ]:
"""Shared notebook functions for the Ciringoni final project."""
import csv
import hashlib
import json
import math
import platform
import random
import shutil
import time
import zipfile
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import yaml

BINS = ['<8', '8–16', '16–32', '32–64', '≥64']
EDGES = [0, 8, 16, 32, 64, np.inf]


def sha256(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def dump(path, data):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(data, indent=2, default=str), encoding='utf-8')


def unpack(path, destination):
    path, destination = Path(path), Path(destination)
    if path.is_dir():
        return path
    digest = sha256(path)
    marker = destination / '.sha256'
    if marker.is_file() and marker.read_text() == digest:
        return destination
    if destination.exists():
        raise FileExistsError(f'Pilih direktori ekstraksi baru untuk {path.name}')
    destination.mkdir(parents=True)
    with zipfile.ZipFile(path) as archive:
        for name in archive.namelist():
            final = (destination / name).resolve()
            if not final.is_relative_to(destination.resolve()):
                raise ValueError(f'Path arsip keluar direktori {name}')
        archive.extractall(destination)
    marker.write_text(digest)
    return destination


def find_input(value, root, kind):
    if value:
        return Path(value)
    candidates = sorted(Path(root).rglob('*.zip'))
    if kind == 'uts':
        candidates = [p for p in candidates if p.name.startswith('UTS_DM_') and 'hasil' in p.name]
    else:
        return None
    if len(candidates) > 1:
        raise ValueError('Beberapa arsip UTS ditemukan. Isi INPUT_UTS dengan satu path.')
    return candidates[0] if candidates else None


def dataset_root(root):
    candidates = [p.parent.parent for p in Path(root).rglob('images/train') if p.is_dir()]
    if len(candidates) != 1:
        raise ValueError(f'Harus ada satu struktur images/train di {root}, ditemukan {len(candidates)}')
    return candidates[0]


def read_labels(path):
    output = []
    for number, line in enumerate(Path(path).read_text(encoding='utf-8-sig').splitlines(), 1):
        if not line.strip():
            continue
        values = list(map(float, line.split()))
        if len(values) != 5 or not all(math.isfinite(v) for v in values):
            raise ValueError(f'Label tidak sah {path} baris {number}')
        c, x, y, w, h = values
        if c != 0 or min(w, h) <= 0 or min(x-w/2, y-h/2) < -1e-5 or max(x+w/2, y+h/2) > 1+1e-5:
            raise ValueError(f'Kelas atau geometri salah {path} baris {number}')
        output.append(values)
    return output


def write_yaml(root):
    data = {'path': str(Path(root).resolve()), 'names': {0: 'infected'}}
    for split in ['train', 'val', 'test']:
        if (Path(root) / 'images' / split).is_dir():
            data[split] = f'images/{split}'
    target = Path(root) / 'data.yaml'
    target.write_text(yaml.safe_dump(data, sort_keys=False), encoding='utf-8')
    return target


def audit_dataset(archive, temp, out):
    extracted = unpack(archive, Path(temp) / 'uts')
    source = dataset_root(extracted)
    root = Path(temp) / 'dataset640'
    if not root.exists():
        shutil.copytree(source, root)
    yaml_path = write_yaml(root)
    split_files = list(extracted.rglob('pembagian_split.csv'))
    if len(split_files) != 1:
        raise ValueError('pembagian_split.csv dari UTS diperlukan untuk audit sumber')
    split_table = pd.read_csv(split_files[0], dtype=str)
    split_table['stem'] = split_table['file'].map(lambda v: Path(v).stem)
    if split_table.stem.duplicated().any():
        raise ValueError('Stem ganda pada pembagian split')
    leakage = int((split_table.groupby('source_id')['split'].nunique() > 1).sum())
    if leakage:
        raise ValueError(f'{leakage} source_id melintasi split')
    records, summary, profiles = [], [], []
    for split in ['train', 'val', 'test']:
        paths = sorted((root / 'images' / split).glob('*.jpg'))
        expected = set(split_table.loc[split_table.split == split, 'stem'])
        if {p.stem for p in paths} != expected:
            raise ValueError(f'Daftar ID tidak sama dengan pembagian_split.csv pada {split}')
        labels = {p.stem for p in (root / 'labels' / split).glob('*.txt')}
        if labels != expected:
            raise ValueError(f'Pasangan label berbeda pada {split}')
        count, backgrounds = 0, 0
        for image in paths:
            with Image.open(image) as im:
                if im.size != (640, 640):
                    raise ValueError(f'Ukuran baseline harus 640 {image.name}')
            boxes = read_labels(root / 'labels' / split / (image.stem + '.txt'))
            records.append({'split': split, 'path': image, 'boxes': boxes})
            count += len(boxes)
            backgrounds += not boxes
            for c, x, y, w, h in boxes:
                side, area = min(w, h)*640, w*h*640**2
                profiles.append({'split': split, 'stem': image.stem, 'sisi_px': side, 'area_px2': area,
                                 'bin': BINS[min(4, np.searchsorted(EDGES, side, side='right')-1)],
                                 'coco': 'small' if area < 1024 else 'medium' if area < 9216 else 'large'})
        summary.append({'split': split, 'gambar': len(paths), 'kotak': count,
                        'sumber': split_table.loc[split_table.split == split, 'source_id'].nunique(),
                        'background': backgrounds})
    pd.DataFrame(summary).to_csv(Path(out) / 'ringkasan_dataset.csv', index=False)
    profile = pd.DataFrame(profiles)
    profile.to_csv(Path(out) / 'profil_ukuran.csv', index=False)
    audit = {'source_leakage': leakage, 'summary': summary, 'input': str(archive)}
    near = list(extracted.rglob('pasangan_hampir_kembar.csv'))
    if near:
        pairs = pd.read_csv(near[0]);mapping = split_table.set_index('stem')['split']
        pairs = pairs[pairs.stem_a.isin(mapping.index) & pairs.stem_b.isin(mapping.index)]
        audit['near_pairs_cross_split'] = int((pairs.stem_a.map(mapping) != pairs.stem_b.map(mapping)).sum())
        if audit['near_pairs_cross_split']:
            raise ValueError('Pasangan hampir kembar melintasi split')
    dump(Path(out) / 'audit_dataset.json', audit)
    return root, yaml_path, records, profile, extracted, split_table


def profile_plot(profile, out):
    import matplotlib.pyplot as plt
    tab = pd.crosstab(profile['split'], profile['bin']).reindex(columns=BINS, fill_value=0)
    percent = tab.div(tab.sum(axis=1), axis=0)*100
    percent.to_csv(Path(out) / 'profil_persen_per_bin.csv')
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for split, group in profile.groupby('split'):
        axes[0].hist(group.sisi_px, bins=np.geomspace(max(.05, profile.sisi_px.min()), 640, 40), alpha=.45, label=split)
    axes[0].set_xscale('log');axes[0].set_xlabel('Sisi terpendek pada acuan 640 (px)');axes[0].legend()
    percent.plot.bar(ax=axes[1]);axes[1].set_ylabel('Persentase kotak');fig.tight_layout()
    fig.savefig(Path(out) / '03_profil_ukuran.png', dpi=140);plt.show()
    tiny = float((profile.sisi_px < 16).mean()*100)
    print(f'{tiny:.2f}% kotak memiliki sisi <16 px pada acuan 640. Prioritas E2 dan tiling diuji pada val.')
    return percent


def montage(records, target, n=6):
    canvas = Image.new('RGB', (960, 700), 'white')
    for i, rec in enumerate(records[:n]):
        with Image.open(rec['path']) as image:
            image = image.convert('RGB').resize((300, 300))
        draw = ImageDraw.Draw(image)
        for c, x, y, w, h in rec['boxes']:
            draw.rectangle([(x-w/2)*300, (y-h/2)*300, (x+w/2)*300, (y+h/2)*300], outline='lime', width=2)
        ox, oy = (i%3)*320, (i//3)*350
        canvas.paste(image, (ox, oy));ImageDraw.Draw(canvas).text((ox+4, oy+304), rec['path'].stem[:36], fill='black')
    canvas.save(target)
    return canvas


def coco_gt(records, split, out):
    result = {'info': {}, 'images': [], 'annotations': [], 'categories': [{'id': 1, 'name': 'infected'}]}
    for rec in [r for r in records if r['split'] == split]:
        iid = len(result['images'])+1
        result['images'].append({'id': iid, 'file_name': rec['path'].name, 'width': 640, 'height': 640})
        for c, x, y, w, h in rec['boxes']:
            result['annotations'].append({'id': len(result['annotations'])+1, 'image_id': iid, 'category_id': 1,
                'bbox': [(x-w/2)*640, (y-h/2)*640, w*640, h*640], 'area': w*h*640**2, 'iscrowd': 0})
    dump(Path(out) / f'coco_{split}_gt.json', result)
    return result


def coco_score(gt, predictions):
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    base = COCO();base.dataset = gt;base.createIndex()
    if predictions:
        dt = base.loadRes(predictions)
    else:
        dt = COCO();dt.dataset = {**gt, 'annotations': []};dt.createIndex()
    score = COCOeval(base, dt, 'bbox');score.evaluate();score.accumulate();score.summarize()
    names = ['AP', 'AP50', 'AP75', 'AP_small', 'AP_medium', 'AP_large', 'AR1', 'AR10', 'AR100']
    return {k: float(v) if v >= 0 else None for k, v in zip(names, score.stats)}


def perfect_test(gt):
    pred = [{k: row[k] for k in ['image_id', 'category_id', 'bbox']} | {'score': 1.0} for row in gt['annotations']]
    # COCO maxDet=100 can cap recall on crowded images. Raise it for this converter unit check.
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    base = COCO();base.dataset = gt;base.createIndex();dt = base.loadRes(pred)
    ev = COCOeval(base, dt, 'bbox')
    limit = max(100, max(Counter(a['image_id'] for a in pred).values(), default=1))
    ev.params.maxDets = [1, 10, limit];ev.evaluate();ev.accumulate()
    valid = ev.eval['precision'][:, :, :, 0, -1]
    ap = float(valid[valid >= 0].mean())
    assert abs(ap-1) < 1e-9, f'Prediksi sempurna gagal {ap}'
    print('Prediksi sempurna lulus, AP =', ap, 'maxDet konversi =', limit)
    return {'AP': ap, 'maxDet_converter_test': limit}


def iou(a, b):
    x, y, w, h = a;xx, yy, ww, hh = b
    inter = max(0, min(x+w, xx+ww)-max(x, xx))*max(0, min(y+h, yy+hh)-max(y, yy))
    return inter / max(1e-12, w*h+ww*hh-inter)


def match_predictions(gt, predictions, conf):
    anns, preds = defaultdict(list), defaultdict(list)
    for a in gt['annotations']:anns[a['image_id']].append(a)
    for p in predictions:
        if p['score'] >= conf:preds[p['image_id']].append(p)
    rows = []
    for image in gt['images']:
        iid = image['id'];used = set()
        for pred in sorted(preds[iid], key=lambda x: -x['score']):
            candidates = [(iou(pred['bbox'], a['bbox']), j) for j, a in enumerate(anns[iid]) if j not in used and a['category_id'] == pred['category_id']]
            best, j = max(candidates, default=(0, -1))
            if best >= .5:
                used.add(j);ann = anns[iid][j];side = min(ann['bbox'][2:])
                rows.append({'file': image['file_name'], 'kind': 'TP', 'side': side, 'score': pred['score'], 'bbox': pred['bbox'], 'gt_bbox': ann['bbox']})
            else:
                rows.append({'file': image['file_name'], 'kind': 'FP', 'side': None, 'score': pred['score'], 'bbox': pred['bbox']})
        for j, a in enumerate(anns[iid]):
            if j not in used:rows.append({'file': image['file_name'], 'kind': 'FN', 'side': min(a['bbox'][2:]), 'score': None, 'bbox': a['bbox']})
    counts = Counter(r['kind'] for r in rows);tp, fp, fn = [counts[k] for k in ['TP', 'FP', 'FN']]
    metrics = {'P_fixed': tp/max(1, tp+fp), 'R_fixed': tp/max(1, tp+fn), 'TP': tp, 'FP': fp, 'FN': fn}
    metrics['F1_fixed'] = 2*tp/max(1, 2*tp+fp+fn)
    for label, lo, hi in zip(['lt8', '8_16', '16_32', '32_64', 'ge64'], EDGES[:-1], EDGES[1:]):
        sizes = [r for r in rows if r['kind'] in ['TP', 'FN'] and lo <= r['side'] < hi]
        metrics['recall_'+label] = sum(r['kind']=='TP' for r in sizes)/len(sizes) if sizes else None
    return metrics, rows


def choose_confidence(gt, predictions, output):
    curve = []
    for c in np.arange(.01, .96, .02):
        m, _ = match_predictions(gt, predictions, float(c));curve.append({'conf': float(c), **m})
    best = max(curve, key=lambda r: (r['F1_fixed'], r['conf']))
    pd.DataFrame(curve).to_csv(Path(output) / 'F1_threshold_val.csv', index=False)
    return best['conf']


def native_data(value, temp, base, out, confirmed=False):
    if not value:
        return None, None, {}
    origin = unpack(value, Path(temp) / 'native_input')
    if (origin / '1280/images/train').exists():
        source = origin / '1280'
    else:
        source = dataset_root(origin)
    audits = list(origin.rglob('audit_pasangan_e2.json'))
    geometry = {}
    if audits:
        audit = json.loads(audits[0].read_text())
        if not audit.get('ready'):
            raise ValueError('Audit native belum ready')
        geometry = {Path(r['filename']).stem: r for r in audit['files'] if r['status'] == 'paired'}
        if any(max(r['source_size']) <= 640 for r in geometry.values()):
            print('Sebagian citra sumber berukuran paling panjang <=640. Detail tambahan terbatas untuk citra itu.')
    elif not confirmed:
        raise ValueError('Ekspor 1280 memerlukan audit asal citra atau KONFIRMASI_NATIVE=True setelah ekspor ulang dari sumber UTS.')
    target = Path(temp) / 'dataset1280'
    if not target.exists():shutil.copytree(source, target)
    paired640 = source.parent / '640'
    if paired640.is_dir() and not (Path(temp) / '640').exists():
        shutil.copytree(paired640, Path(temp) / '640')
    checks = []
    for split in ['train', 'val']:
        paths = sorted((target / 'images' / split).glob('*.jpg'))
        expected = {p.stem for p in (base / 'images' / split).glob('*.jpg')}
        if expected != {p.stem for p in paths}:raise ValueError(f'ID native {split} berbeda')
        for p in paths:
            with Image.open(p) as im:
                if im.size != (1280, 1280):raise ValueError(f'Ukuran native salah {p}')
            a = read_labels(base / 'labels' / split / (p.stem+'.txt'))
            b = read_labels(target / 'labels' / split / (p.stem+'.txt'))
            if len(a) != len(b):raise ValueError(f'Jumlah kotak native berbeda {p.name}')
            if p.stem in geometry:
                r = geometry[p.stem]
                projected = []
                for c,x,y,w,h in b:
                    box = project_box([(x-w/2)*1280,(y-h/2)*1280,w*1280,h*1280],p.stem,1280,geometry)
                    projected.append([c,(box[0]+box[2]/2)/640,(box[1]+box[3]/2)/640,box[2]/640,box[3]/640])
                if not np.allclose(np.array(a),np.array(projected),atol=1e-5):raise ValueError(f'Proyeksi label berbeda {p.name}')
            elif a and not np.allclose(np.array(a),np.array(b),atol=.004):
                raise ValueError(f'Koordinat native berbeda terlalu jauh {p.name}')
        checks.append({'split': split, 'images': len(paths)})
    dump(Path(out) / 'audit_native_kaggle.json', {'checks': checks, 'geometry_manifest': bool(geometry), 'confirmed_source_export': confirmed})
    return target, write_yaml(target), geometry


def project_box(box, stem, size, geometry):
    x,y,w,h = box
    if size == 1280 and stem in geometry:
        a,b = geometry[stem]['content_at_1280'],geometry[stem]['content_at_640']
        return [(x-a[0])*b[2]/a[2]+b[0],(y-a[1])*b[3]/a[3]+b[1],w*b[2]/a[2],h*b[3]/a[3]]
    return [v*640/size for v in box]


def export_tiles(native, temp, out, tile=640, overlap=.2, visibility=.5):
    root = Path(temp) / 'tiles_native640'
    if root.exists():raise FileExistsError('Ekspor tile sudah ada. Gunakan TEMP baru agar konfigurasi terjaga.')
    (root/'images/train').mkdir(parents=True);(root/'labels/train').mkdir(parents=True)
    rng = random.Random(42);positive,empty,evidence = [],[],[];counts = Counter()
    stride = round(tile*(1-overlap))
    for image in sorted((native/'images/train').glob('*.jpg')):
        labels = read_labels(native/'labels/train'/(image.stem+'.txt'))
        with Image.open(image) as im:
            width,height = im.size
            xs = sorted(set(list(range(0,width-tile+1,stride))+[width-tile]))
            ys = sorted(set(list(range(0,height-tile+1,stride))+[height-tile]))
            for oy in ys:
                for ox in xs:
                    boxes = []
                    for c,x,y,w,h in labels:
                        x0,y0,x1,y1 = (x-w/2)*width,(y-h/2)*height,(x+w/2)*width,(y+h/2)*height
                        a,b,cc,dd = max(x0,ox),max(y0,oy),min(x1,ox+tile),min(y1,oy+tile)
                        if cc<=a or dd<=b or (cc-a)*(dd-b) < visibility*(x1-x0)*(y1-y0):continue
                        boxes.append([c,((a+cc)/2-ox)/tile,((b+dd)/2-oy)/tile,(cc-a)/tile,(dd-b)/tile])
                    item=(image,ox,oy,boxes)
                    (positive if boxes else empty).append(item)
    chosen = positive+rng.sample(empty,min(len(empty),len(positive)//3))
    for image,ox,oy,boxes in chosen:
        stem=f'{image.stem}_x{ox}_y{oy}'
        path=root/'images/train'/(stem+'.jpg')
        with Image.open(image) as im:im.crop((ox,oy,ox+tile,oy+tile)).save(path,quality=95)
        label=root/'labels/train'/(stem+'.txt')
        label.write_text(''.join(' '.join(f'{v:.8f}' for v in row)+'\n' for row in boxes))
        read_labels(label)
        if boxes and len(evidence)<6:evidence.append({'path':path,'boxes':boxes})
        counts.update(tiles=1,boxes=len(boxes),empty=not boxes)
    data={'path':str(root),'train':'images/train','val':str(native/'images/val'),'names':{0:'infected'}}
    if (native/'images/test').is_dir():data['test']=str(native/'images/test')
    yaml_path=root/'data.yaml';yaml_path.write_text(yaml.safe_dump(data))
    dump(Path(out)/'audit_tile_native.json',{'tile':tile,'overlap':overlap,'visibility':visibility,**counts})
    montage(evidence,Path(out)/'08_enam_tile.png')
    return yaml_path


def load_yolo(path):
    from ultralytics import YOLO
    return YOLO(str(path))


def checkpoint_path(run, checkpoints, input_root):
    if checkpoints.get(run):
        p=Path(checkpoints[run])
        if p.exists():return p
        raise FileNotFoundError(p)
    matches=list(Path(input_root).rglob(f'{run}/weights/best.pt'))
    if len(matches)==1:return matches[0]
    raise FileNotFoundError(f'Isi CHECKPOINTS[{run!r}] atau tambahkan dataset bobot run {run}')


def train_or_load(run, initial, yaml_path, size, config, extra=None):
    if not config['LATIH'].get(run,False):
        return checkpoint_path(run,config['CHECKPOINTS'],config['INPUT_ROOT'])
    import torch
    if not torch.cuda.is_available():raise RuntimeError('Aktifkan GPU untuk training')
    project=Path(config['OUT'])/'runs';target=project/run
    resume=config['RESUME'].get(run)
    if target.exists() and not resume:raise FileExistsError(f'Run sudah ada {target}')
    options=dict(data=str(yaml_path),imgsz=size,epochs=config['EPOCHS'],patience=config['PATIENCE'],batch=config['BATCH'],
        seed=config['SEED'],deterministic=True,device=0,workers=2,project=str(project),name=run,
        mosaic=1.,close_mosaic=0,optimizer='AdamW',lr0=.002)
    options.update(extra or {})
    model=load_yolo(resume or initial)
    if resume:model.train(resume=True)
    else:model.train(**options)
    if model.trainer.batch_size != config['BATCH']:raise RuntimeError('Batch berubah. Samakan batch pada pasangan eksperimen.')
    return Path(model.trainer.save_dir)/'weights/best.pt'


def predict_coco(model, gt, root, size, geometry, device=0, sahi_options=None):
    rows=[]
    if sahi_options:
        from sahi import AutoDetectionModel
        from sahi.predict import get_sliced_prediction
        detector=AutoDetectionModel.from_pretrained(model_type='ultralytics',model_path=str(model),confidence_threshold=.001,device='cuda:0' if device==0 else 'cpu')
        detector.image_size=sahi_options['tile']
    else:detector=load_yolo(model)
    for image in gt['images']:
        path=Path(root)/image['file_name']
        if sahi_options:
            r=get_sliced_prediction(str(path),detector,slice_height=sahi_options['tile'],slice_width=sahi_options['tile'],
                overlap_height_ratio=sahi_options['overlap'],overlap_width_ratio=sahi_options['overlap'],
                perform_standard_pred=sahi_options['standard'],postprocess_type='NMS',postprocess_match_metric='IOU',postprocess_match_threshold=.5,verbose=0)
            raw=[(o.bbox.to_xywh(),o.score.value,int(o.category.id)) for o in r.object_prediction_list]
        else:
            r=detector.predict(str(path),imgsz=size,device=device,conf=.001,max_det=100,verbose=False)[0]
            raw=[([a,b,c-a,d-b],score,int(klass)) for (a,b,c,d),score,klass in zip(r.boxes.xyxy.cpu().tolist(),r.boxes.conf.cpu().tolist(),r.boxes.cls.cpu().tolist())]
        for box,score,klass in sorted(raw,key=lambda r:-r[1])[:100]:
            mapped=project_box(box,path.stem,size,geometry)
            x,y,w,h=mapped;x1,y1=np.clip([x,y],0,640);x2,y2=np.clip([x+w,y+h],0,640)
            if x2>x1 and y2>y1:rows.append({'image_id':image['id'],'category_id':klass+1,'bbox':[float(x1),float(y1),float(x2-x1),float(y2-y1)],'score':float(score)})
    return rows


def cpu_latency(model_path,image,size,out,sahi_options=None,warmup=10,repeats=100):
    import torch
    torch.set_num_threads(4)
    if sahi_options:
        from sahi import AutoDetectionModel
        from sahi.predict import get_sliced_prediction
        model=AutoDetectionModel.from_pretrained(model_type='ultralytics',model_path=str(model_path),confidence_threshold=.05,device='cpu')
        model.image_size=sahi_options['tile']
        rgb=np.array(Image.open(image).convert('RGB'))
        def infer():
            return get_sliced_prediction(rgb,model,slice_height=sahi_options['tile'],slice_width=sahi_options['tile'],
                overlap_height_ratio=sahi_options['overlap'],overlap_width_ratio=sahi_options['overlap'],
                perform_standard_pred=sahi_options['standard'],postprocess_type='NMS',postprocess_match_metric='IOU',postprocess_match_threshold=.5,verbose=0)
    else:
        model=load_yolo(model_path);rgb=np.array(Image.open(image).convert('RGB'))[:,:,::-1].copy()
        def infer():return model.predict(rgb,imgsz=size,device='cpu',batch=1,conf=.05,max_det=100,verbose=False)
    for _ in range(warmup):infer()
    torch.set_num_threads(4)
    times=[]
    for _ in range(repeats):
        start=time.perf_counter();infer();times.append((time.perf_counter()-start)*1000)
    result={'latensi_cpu_ms':float(np.median(times)),'latensi_cpu_p90_ms':float(np.percentile(times,90)),
        'cpu_threads':4,'warmup':warmup,'ulang':repeats,'cpu':platform.processor(),'platform':platform.platform(),
        'scope':'wall clock sesudah decoding, preprocessing sampai postprocessing, batch 1','image':Path(image).name}
    dump(Path(out)/'latensi.json',result)
    return result


def catat(run, result, out):
    path=Path(out)/'eksperimen.csv'
    old=pd.read_csv(path) if path.exists() else pd.DataFrame()
    if 'id' in old:old=old[old.id != run]
    row={'id':run,**result}
    merged=pd.concat([old,pd.DataFrame([row])],ignore_index=True)
    stage=path.with_suffix('.tmp');merged.to_csv(stage,index=False);stage.replace(path)
    return row


def evaluate_run(run,checkpoint,root,yaml_path,size,gt,config,geometry=None,sahi_options=None,split='val',fixed_conf=None):
    dest=Path(config['OUT'])/run;dest.mkdir(parents=True,exist_ok=True)
    predictions=predict_coco(checkpoint,gt,root,size,geometry or {},config['DEVICE'],sahi_options)
    dump(dest/f'coco_{split}_prediksi.json',predictions)
    scored=coco_score(gt,predictions)
    conf=choose_confidence(gt,predictions,dest) if split=='val' else fixed_conf
    if conf is None:raise ValueError('Confidence operasional dari val wajib diberikan sebelum test')
    fixed,rows=match_predictions(gt,predictions,conf)
    pd.DataFrame(rows).to_csv(dest/'TP_FP_FN.csv',index=False)
    model=load_yolo(checkpoint)
    metrics={}
    if not sahi_options:
        v=model.val(data=str(yaml_path),split=split,imgsz=size,batch=config['BATCH'],device=config['DEVICE'],workers=0,
                    plots=True,project=str(dest),name='ultralytics',exist_ok=True,max_det=100)
        metrics={'P':float(v.box.mp),'R':float(v.box.mr),'mAP50':float(v.box.map50),'mAP50_95':float(v.box.map)}
        metrics['F1']=2*metrics['P']*metrics['R']/max(1e-12,metrics['P']+metrics['R'])
    from ultralytics.utils.torch_utils import get_flops
    is_torch_model = callable(getattr(model.model, 'parameters', None))
    try:flops=float(get_flops(model.model,imgsz=size)) if is_torch_model else None
    except (RuntimeError,TypeError):flops=None
    weight=Path(checkpoint)
    weight_bytes = weight.stat().st_size if weight.is_file() else sum(p.stat().st_size for p in weight.rglob('*') if p.is_file())
    result={'split':split,'perlakuan':config['DESCRIPTIONS'].get(run,run),'model':str(checkpoint),'imgsz':size,
        'seed':config['SEED'],'epochs':config['EPOCHS'],'tanggal':datetime.now().isoformat(),
        **metrics,**scored,**fixed,'conf_operasional':conf,'params_M':sum(p.numel() for p in model.model.parameters())/1e6 if is_torch_model else None,
        'gflops':flops,'ukuran_MB':weight_bytes/1e6,
        'protokol':'COCO pada acuan 640, maxDet100, conf prediksi0.001','status':'dihitung pada eksekusi ini'}
    if config['UKUR_CPU']:
        image=Path(root)/gt['images'][0]['file_name']
        result.update(cpu_latency(checkpoint,image,size,dest,sahi_options))
    catat(run,result,config['OUT']);dump(dest/'metrik.json',result)
    return result,rows


def error_analysis(rows,base,uts_root,out):
    out=Path(out);table=pd.DataFrame(rows);table['stem']=table.file.map(lambda x:Path(x).stem)
    files=list(Path(uts_root).rglob('inventaris.csv'))
    if files:
        inv=pd.read_csv(files[0]);inv=inv.drop_duplicates('stem')
        table=table.merge(inv,on='stem',how='left',suffixes=('','_uts'))
    table.to_csv(out/'analisis_error.csv',index=False)
    for kind,n in [('FN',12),('FP',12),('TP',6)]:
        selected=table[table.kind==kind].sort_values('side' if kind=='FN' else 'score',ascending=(kind=='FN')).head(n)
        records=[]
        for _,r in selected.iterrows():
            x,y,w,h=r['bbox'];records.append({'path':base/'images/test'/r.file,'boxes':[[0,(x+w/2)/640,(y+h/2)/640,w/640,h/640]]})
        if records:
            # Two contact sheets for twelve examples, six per image.
            for offset in range(0,len(records),6):montage(records[offset:offset+6],out/f'13_{kind.lower()}_{offset//6+1}.png')
    for key in ['pengunggah','source_id','blur_score']:
        if key not in table:continue
        groups=pd.qcut(table[key],4,duplicates='drop').astype(str) if key=='blur_score' else table[key]
        agg=pd.crosstab(groups,table.kind).reindex(columns=['TP','FN','FP'],fill_value=0)
        agg['recall']=agg.TP/(agg.TP+agg.FN).replace(0,np.nan);agg.to_csv(out/f'recall_{key}.csv')
        import matplotlib.pyplot as plt
        agg['recall'].plot.bar(figsize=(12,4));plt.tight_layout();plt.savefig(out/f'13_recall_{key}.png',dpi=140);plt.close()
    template=table[['file','kind']].copy();template['penyebab_manual']='';template.to_csv(out/'penyebab_error_isi_manual.csv',index=False)


def compare_reference_labels(gt, predictions, class_folder, reference_folder, conf, out):
    """Compare both label sets after they have been transformed into reference640 coordinates."""
    import copy
    common = {p.stem for p in Path(class_folder).glob('*.txt')} & {p.stem for p in Path(reference_folder).glob('*.txt')}
    images = [im for im in gt['images'] if Path(im['file_name']).stem in common]
    if not images:
        print('Tidak ada ID evaluasi yang memiliki kedua himpunan label640.')
        return
    image_ids = {im['id'] for im in images}
    selected = [p for p in predictions if p['image_id'] in image_ids]
    results = []
    for name, folder in [('kelas', class_folder), ('acuan', reference_folder)]:
        comparison = {'info': {}, 'images': images, 'categories': copy.deepcopy(gt['categories']), 'annotations': []}
        for im in images:
            for c,x,y,w,h in read_labels(Path(folder)/(Path(im['file_name']).stem+'.txt')):
                comparison['annotations'].append({'id':len(comparison['annotations'])+1,'image_id':im['id'],
                    'category_id':1,'bbox':[(x-w/2)*640,(y-h/2)*640,w*640,h*640], 'area':w*h*640**2,'iscrowd':0})
        scores = coco_score(comparison, selected)
        fixed,_ = match_predictions(comparison, selected, conf)
        results.append({'label':name,'images':len(images),**scores,**fixed})
    pd.DataFrame(results).to_csv(Path(out)/'13_perbandingan_label_acuan.csv',index=False)
    print(pd.DataFrame(results).to_string(index=False))


def package(out,group,final_checkpoint=None,exports=None):
    out=Path(out)
    if final_checkpoint and Path(final_checkpoint).is_file():shutil.copy2(final_checkpoint,out/'best.pt')
    for export in exports or []:
        p=Path(export)
        if p.is_dir():shutil.copytree(p,out/'model_ekspor'/p.name,dirs_exist_ok=True)
        else:(out/'model_ekspor').mkdir(exist_ok=True);shutil.copy2(p,out/'model_ekspor'/p.name)
    path=out.parent/f'{group}_hasil.zip'
    with zipfile.ZipFile(path,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in out.rglob('*'):
            if p.is_file():z.write(p,p.relative_to(out))
    return path

random.seed(SEED);np.random.seed(SEED)
if EVALUASI or any(LATIH.values()) or JALANKAN_TEST or JALANKAN_INT8:
    import importlib.metadata
    for package, version in [('ultralytics','8.4.210'), ('sahi','0.14.0')]:
        try:installed=importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:installed=None
        if installed!=version:subprocess.check_call([sys.executable,'-m','pip','install','-q',f'{package}=={version}'])
    import torch,ultralytics,sahi
    torch.manual_seed(SEED)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
    dump(OUT/'lingkungan.json',{'python':sys.version,'torch':torch.__version__,'ultralytics':ultralytics.__version__,
        'sahi':sahi.__version__,'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})
CONFIG={k:globals()[k] for k in ['LATIH','CHECKPOINTS','RESUME','INPUT_ROOT','OUT','EPOCHS','PATIENCE','BATCH','SEED','DEVICE','UKUR_CPU']}
CONFIG['DESCRIPTIONS']={'E0':'YOLO26n baseline640','E1':'YOLO26s kapasitas640','E2':'YOLO26n sumber native1280',
 'E2_control_native640':'YOLO26n sumber native640 kontrol pasangan','E3':'YOLO26n train tile native640, val utuh1280',
 'E5':'YOLO26n head P2','E6':'YOLO26n tanpa mosaic'}
HASIL_VAL={};MODEL_PATHS={};EXPORTS=[];FINAL_PATH=None
BASE=None;NATIVE=None;GEO={};TILE_YAML=None;GT_VAL=None
print('Fungsi analisis, training, E2, E3, SAHI dan pencatatan siap.')

## 2. Memuat dan memverifikasi dataset UTS

Folder `/kaggle/input` hanya-baca, sedangkan Ultralytics perlu menulis file cache label. Karena itu dataset disalin dulu ke folder yang bisa ditulis.

**Langkah:**

1. Temukan folder `dataset/` di dalam input hasil UTS (berisi `images/{train,val,test}`, `labels/{train,val,test}`, `data.yaml`). Salin ke `TEMP / "dataset640"`.
2. Tulis ulang `data.yaml` dengan kunci `path:` absolut ke folder salinan, agar tidak bergantung pada folder kerja.
3. Hitung jumlah gambar, file label, dan kotak per split. Pastikan setiap gambar punya file label (gambar latar belakang boleh punya file label kosong).
4. Baca `pembagian_split.csv` dari UTS. Cocokkan isi setiap folder split dengan kolom `split` di CSV itu. **Harus identik**; bila tidak, berhenti dan perbaiki.
5. Periksa kebocoran: tidak boleh ada `source_id` yang muncul di lebih dari satu split.
6. Tampilkan 6 gambar acak dari `train` beserta kotaknya untuk memastikan label terbaca dengan benar.

**Hasil yang diharapkan:** tabel ringkasan per split (gambar, kotak, sumber, gambar latar belakang) disimpan sebagai `ringkasan_dataset.csv`.

In [ ]:
uts=find_input(INPUT_UTS, INPUT_ROOT, 'uts')
if uts:
    BASE, BASE_YAML, RECORDS, PROFILE, UTS_ROOT, SPLITS = audit_dataset(uts,TEMP,OUT)
    display(pd.read_csv(OUT/'ringkasan_dataset.csv'))
else:
    print('Tambahkan ZIP hasil UTS melalui Add Input, lalu jalankan kembali Bagian 2.')

In [ ]:
if BASE:
    train=[r for r in RECORDS if r['split']=='train']
    examples=random.Random(SEED).sample(train,min(6,len(train)))
    display(montage(examples,OUT/'02_enam_train.png'))

In [ ]:
if BASE:
    print(json.dumps(json.loads((OUT/'audit_dataset.json').read_text()),indent=2))
    audit=json.loads((OUT/'audit_dataset.json').read_text())
    audit['notebook_sumber_kaggle']=NOTEBOOK_SUMBER_KAGGLE
    audit['sha256_arsip_input']=sha256(uts) if Path(uts).is_file() else None
    dump(OUT/'audit_dataset.json',audit)

## 3. Profil ukuran objek

Sebelum melatih, ukur seberapa kecil objek yang harus dideteksi. Hasilnya menentukan prioritas eksperimen di Bagian 7–11.

**Langkah:**

1. Untuk setiap kotak di semua split, hitung lebar dan tinggi dalam piksel: `w_px = w * IMG_SIZE`, `h_px = h * IMG_SIZE`, sisi terpendek `s = min(w_px, h_px)`, dan luas `a = w_px * h_px`.
2. Kelompokkan berdasarkan sisi terpendek ke dalam bin: `< 8`, `8–16`, `16–32`, `32–64`, `≥ 64` px (mengikuti kategori AI-TOD dan COCO).
3. Kelompokkan juga berdasarkan **luas** sesuai COCO: small (`a < 32²`), medium (`32² ≤ a < 96²`), large (`a ≥ 96²`). Kategori inilah yang dipakai COCOeval untuk AP-small.
4. Buat tabel persentase per bin **untuk setiap split** dan histogram sisi terpendek (skala log pada sumbu x).
5. Hitung juga jumlah kotak per gambar (median dan maksimum).
6. Simpan tabel sebagai `profil_ukuran.csv` dan gambar sebagai `03_profil_ukuran.png`.

**Tulis interpretasi (sel markdown):** berapa persen kotak < 16 px? Apakah distribusi ukuran train, val, dan test mirip? Eksperimen mana yang menjadi prioritas?

In [ ]:
if BASE:
    display(profile_plot(PROFILE,OUT))

In [ ]:
if BASE:
    per_image=pd.DataFrame([{'split':r['split'],'kotak':len(r['boxes'])} for r in RECORDS])
    display(per_image.groupby('split').kotak.agg(['median','max']))
    display(pd.crosstab(PROFILE.split,PROFILE.coco))

**Interpretasi kelompok.** UTS mencatat 937 kotak dengan sisi <8 px pada acuan 640. Persentase <16 px dan perbedaan train/val/test dihitung otomatis pada Bagian 3 setelah ZIP ditambahkan. Resolusi native dan tiling menjadi prioritas pengujian M2. AP_small COCO menggunakan luas pada acuan 640 yang konsisten antar-run.

## 4. Alat evaluasi

Semua eksperimen dievaluasi dengan fungsi yang sama. Bangun fungsi-fungsi berikut **sekali** di bagian ini, lalu pakai ulang.

### 4.1 Metrik Ultralytics

1. Tulis fungsi `eval_ultralytics(model, split, imgsz)` yang menjalankan `model.val(data=..., split=split, imgsz=imgsz, batch=..., plots=True)`.
2. Kembalikan dict berisi: presisi (`box.mp`), recall (`box.mr`), `mAP50` (`box.map50`), `mAP50-95` (`box.map`), dan waktu per tahap (`speed`).
3. Hitung F1 dari presisi dan recall.

### 4.2 AP per ukuran dengan COCOeval

Ultralytics tidak melaporkan AP-small secara bawaan untuk dataset kustom.

1. Tulis fungsi yang mengubah label YOLO satu split menjadi **COCO JSON ground truth**: `images` (id, file_name, width, height), `annotations` (id, image_id, category_id, `bbox = [x_min, y_min, w, h]` dalam piksel, `area = w*h`, `iscrowd = 0`), dan `categories`.
2. Tulis fungsi yang menjalankan `model.predict` pada semua gambar split tersebut dengan **confidence rendah** (misalnya `conf=0.001`) dan menyimpan hasilnya sebagai **COCO JSON prediksi**: daftar `{image_id, category_id, bbox, score}`.
3. Jalankan `pycocotools` `COCOeval(gt, dt, "bbox")` → `evaluate()`, `accumulate()`, `summarize()`.
4. Ambil `stats[0]` (AP), `stats[1]` (AP50), `stats[3]` (AP-small), `stats[4]` (AP-medium), `stats[5]` (AP-large), `stats[8]` (AR@100). Nilai `-1` berarti tidak ada objek pada kategori itu.
5. Uji fungsi ini dengan "prediksi sempurna" (ground truth dijadikan prediksi, skor 1). Hasilnya harus AP = 1. Bila tidak, ada kesalahan konversi koordinat.

### 4.3 Recall per bin ukuran

1. Pilih **ambang confidence operasional** dari kurva F1 pada split `val` (ambang dengan F1 tertinggi). Ambang ini ditetapkan per model, hanya dengan data val.
2. Untuk setiap gambar, cocokkan prediksi (di atas ambang) dengan ground truth memakai IoU ≥ 0,5 secara greedy (confidence tertinggi dulu). Fungsi `iou` dan `cocokkan` dari notebook UTS boleh dipakai ulang.
3. Hitung recall untuk setiap bin sisi terpendek dari Bagian 3.
4. Simpan juga daftar **FN** dan **FP** per gambar; daftar ini dipakai di Bagian 13.

### 4.4 Latensi dan ukuran model

1. Tulis fungsi `ukur_latensi(model_path, device="cpu", imgsz, n_warmup=10, n_ulang=100)` yang memakai **batch 1** dan gambar yang sama, lalu mengembalikan median dan persentil-90 dalam milidetik.
2. Catat jumlah parameter dan GFLOPs (misalnya dari `model.info()`) serta ukuran file bobot (MB).
3. Ukur latensi CPU di jenis mesin yang sama untuk semua model, dan tulis spesifikasinya.

In [ ]:
if BASE:
    GT_VAL=coco_gt(RECORDS,'val',OUT)
    print('GT val',len(GT_VAL['images']),'gambar',len(GT_VAL['annotations']),'anotasi')

In [ ]:
if GT_VAL:
    dump(OUT/'uji_prediksi_sempurna.json',perfect_test(GT_VAL))

In [ ]:
print('Recall per bin dihitung greedy IoU >=0.5. Threshold dipilih dari kurva F1 val dan disimpan per run.')
print('Kolom P/R Ultralytics dan P_fixed/R_fixed memiliki threshold evaluasi yang berbeda dan disimpan terpisah.')

In [ ]:
print('Benchmark baru memakai gambar val pertama yang sama, batch1, CPU4 thread, wall clock sesudah decode, median dan p90.')
print('Benchmark historis repo memakai median/p95 lintas356 gambar. Gunakan hanya perbandingan dengan protokol yang sama.')

## 5. Protokol eksperimen dan pencatatan

**Langkah:**

1. Buat (atau muat, bila sudah ada di input) tabel `eksperimen.csv` dengan kolom minimal:

   `id, perlakuan, model, imgsz, epochs, seed, tanggal, P, R, F1, conf_operasional, mAP50, mAP50_95, AP_small, AP_medium, AR100, recall_lt8, recall_8_16, recall_16_32, recall_32_64, recall_ge64, latensi_cpu_ms, params_M, gflops, ukuran_MB, catatan`

2. Tulis fungsi `catat(id, hasil)` yang menambahkan atau memperbarui satu baris, lalu langsung menyimpan CSV.
3. Tulis fungsi `latih(id, model_awal, data_yaml, imgsz, **kwargs)` yang:
   - memakai `project=WORK/"runs"`, `name=id`, `seed=SEED`, `epochs=EPOCHS`, `patience=PATIENCE`;
   - melewati pelatihan bila `LATIH[id]` bernilai `False` dan memuat `best.pt` dari input;
   - mendukung `resume=True` bila sesi Kaggle terputus (melanjutkan dari `last.pt`).
4. Isi tabel rencana eksperimen di bawah sebelum mulai melatih.

| ID | Perlakuan | Faktor yang diubah | Hipotesis kelompok |
|---|---|---|---|
| E0 | YOLO26n, imgsz 640, pengaturan default | — | Diisi pada tabel rencana yang dihasilkan sel berikut |
| E1 | YOLO26s, imgsz 640 | kapasitas model | Diisi pada tabel rencana yang dihasilkan sel berikut |
| E2 | imgsz 1024 atau 1280 (dataset diekspor ulang) | resolusi | Diisi pada tabel rencana yang dihasilkan sel berikut |
| E3 | latih pada potongan (tile) citra resolusi tinggi | data latih | Diisi pada tabel rencana yang dihasilkan sel berikut |
| E4 | SAHI: inferensi terpotong pada model terbaik | cara inferensi | Diisi pada tabel rencana yang dihasilkan sel berikut |
| E5 | tambah head P2 (stride 4) | arsitektur | Diisi pada tabel rencana yang dihasilkan sel berikut |
| E6 | augmentasi (copy-paste, scale, mosaic, dll.) | augmentasi | Diisi pada tabel rencana yang dihasilkan sel berikut |
| E7 | ekspor INT8 model terbaik | deployment | Diisi pada tabel rencana yang dihasilkan sel berikut |

In [ ]:
HISTORY=[{'id': 'E0', 'perlakuan': 'Baseline YOLO26n, imgsz 640', 'split': 'val', 'presisi': '0.19092522827817351', 'recall': '0.17505668934240362', 'mAP50': '0.07998465358889204', 'mAP50-95': '0.022258051549890776', 'AP-small': '0.023398142434680896', 'latensi_cpu_ms': '', 'sumber_csv': 'hasil_m1/eksperimen.csv', 'status': 'hasil lokal terarsip di repo e376ec2'}, {'id': 'E1', 'perlakuan': 'YOLO26n, imgsz 640, freeze=10', 'split': 'val', 'presisi': '0.19244226593287161', 'recall': '0.1346938775510204', 'mAP50': '0.08145873014673449', 'mAP50-95': '0.024039291108406052', 'AP-small': '0.024280985130644247', 'latensi_cpu_ms': '', 'sumber_csv': 'hasil_m1/eksperimen.csv', 'status': 'hasil lokal terarsip di repo e376ec2'}, {'id': 'E2_control_native_640', 'split': 'val', 'gambar': '356', 'imgsz': '640', 'val_ultralytics_batch': '4', 'checkpoint': 'hasil_m2\\eksperimen_e2_native\\runs\\E2_control_native_640\\weights\\best.pt', 'presisi': '0.1558599678751565', 'recall': '0.1582766439909297', 'mAP50': '0.06586806058433652', 'mAP50-95': '0.01986461600607105', 'AP-small': '0.021893977734593784', 'presisi_conf_0.05_iou_0.5': '0.10329817512137954', 'recall_conf_0.05_iou_0.5': '0.27981859410430837', 'cpu_median_ms': '28.613749999749416', 'cpu_p95_ms': '39.16857500007609', 'cpu_torch_threads': '4', 'sumber_csv': 'hasil_m2/evaluasi/eksperimen_e2_val.csv', 'status': 'hasil lokal terarsip di repo e376ec2'}, {'id': 'E2_native_1280', 'split': 'val', 'gambar': '356', 'imgsz': '1280', 'val_ultralytics_batch': '4', 'checkpoint': 'hasil_m2\\eksperimen_e2_native\\runs\\E2_native_1280\\weights\\best.pt', 'presisi': '0.19390055409376167', 'recall': '0.19410430839002268', 'mAP50': '0.1032722828065831', 'mAP50-95': '0.03261961670183809', 'AP-small': '0.04268878263417472', 'presisi_conf_0.05_iou_0.5': '0.1277715744137821', 'recall_conf_0.05_iou_0.5': '0.363265306122449', 'cpu_median_ms': '120.37474999942788', 'cpu_p95_ms': '146.4887999986786', 'cpu_torch_threads': '4', 'sumber_csv': 'hasil_m2/evaluasi/eksperimen_e2_val.csv', 'status': 'hasil lokal terarsip di repo e376ec2'}, {'id': 'E0_640', 'split': 'val', 'gambar': '356', 'presisi': '0.17056643445667474', 'recall': '0.17188208616780046', 'mAP50': '0.0802034519118993', 'mAP50-95': '0.023534039392602155', 'AP-small': '0.02408517721828604', 'presisi_conf_0.05_iou_0.5': '0.11226679874525343', 'recall_conf_0.05_iou_0.5': '0.30839002267573695', 'cpu_median_ms': '36.979149997932836', 'cpu_p95_ms': '51.036900000781316', 'cpu_torch_threads': '4', 'checkpoint': '', 'sumber_csv': 'hasil_m2/evaluasi/eksperimen_mosaic_val.csv', 'status': 'hasil lokal terarsip di repo e376ec2'}, {'id': 'E6_640_no_mosaic', 'split': 'val', 'gambar': '356', 'presisi': '0.19092522827817351', 'recall': '0.17505668934240362', 'mAP50': '0.07998465358889204', 'mAP50-95': '0.022258051549890776', 'AP-small': '0.023398142434680896', 'presisi_conf_0.05_iou_0.5': '0.12085731295892042', 'recall_conf_0.05_iou_0.5': '0.2761904761904762', 'cpu_median_ms': '36.029499999131076', 'cpu_p95_ms': '56.269599999268394', 'cpu_torch_threads': '4', 'checkpoint': '', 'sumber_csv': 'hasil_m2/evaluasi/eksperimen_mosaic_val.csv', 'status': 'hasil lokal terarsip di repo e376ec2'}, {'id': 'E5_640_p2', 'split': 'val', 'gambar': '356', 'presisi': '0.12405040177520417', 'recall': '0.16145124716553289', 'mAP50': '0.051473086695553324', 'mAP50-95': '0.014047448753952743', 'AP-small': '0.01719756787024344', 'presisi_conf_0.05_iou_0.5': '0.10518934081346423', 'recall_conf_0.05_iou_0.5': '0.272108843537415', 'cpu_median_ms': '44.06880000078672', 'cpu_p95_ms': '59.152199999971344', 'cpu_torch_threads': '4', 'checkpoint': '', 'sumber_csv': 'hasil_m2/evaluasi/eksperimen_mosaic_val.csv', 'status': 'hasil lokal terarsip di repo e376ec2'}, {'id': 'E3_control_full_640', 'split': 'val', 'gambar': '356', 'presisi': '0.10194805092778472', 'recall': '0.10702947845804989', 'mAP50': '0.032133069563529904', 'mAP50-95': '0.00838382726041426', 'AP-small': '0.014290212428987682', 'presisi_conf_0.05_iou_0.5': '0.049199602062042144', 'recall_conf_0.05_iou_0.5': '0.24671201814058957', 'cpu_median_ms': '', 'cpu_p95_ms': '', 'cpu_torch_threads': '', 'checkpoint': 'eksperimen_e3_640_verified/runs/E3_control_full_640/weights/last.pt', 'sumber_csv': 'hasil_m2/evaluasi/eksperimen_mosaic_val.csv', 'status': 'hasil lokal terarsip di repo e376ec2'}, {'id': 'E3_tile384_640', 'split': 'val', 'gambar': '356', 'presisi': '0.017144695489346863', 'recall': '0.18095238095238095', 'mAP50': '0.011483736054843714', 'mAP50-95': '0.0024019727387257815', 'AP-small': '0.00339109766581214', 'presisi_conf_0.05_iou_0.5': '0.026493506493506493', 'recall_conf_0.05_iou_0.5': '0.11564625850340136', 'cpu_median_ms': '', 'cpu_p95_ms': '', 'cpu_torch_threads': '', 'checkpoint': 'eksperimen_e3_640_verified/runs/E3_tile384_640/weights/last.pt', 'sumber_csv': 'hasil_m2/evaluasi/eksperimen_mosaic_val.csv', 'status': 'hasil lokal terarsip di repo e376ec2'}]
historical=pd.DataFrame(HISTORY)
historical.to_csv(OUT/'eksperimen_historis_repo.csv',index=False)
display(historical[['id','mAP50-95','AP-small','sumber_csv']])
e4_summary=pd.DataFrame([
 {'id':'E4_full_local','AP_small':.04265,'recall_conf005_iou05':.3628,'cpu_median_ms':120.0},
 {'id':'E4_sahi_local','AP_small':.03899,'recall_conf005_iou05':.3732,'cpu_median_ms':307.7}])
e4_summary['sumber']='HANDOFF.md commit e376ec2, ringkasan tanpa JSON rinci pada repo publik'
e4_summary.to_csv(OUT/'ringkasan_e4_handoff.csv',index=False)
display(e4_summary)
# ID lokal lama dipertahankan hanya pada tabel historis, setiap run baru masuk eksperimen.csv.

In [ ]:
plan=[
 ('E0','Baseline n640','Acuan pengaturan bersama'),('E1','s640','Kapasitas lebih besar dapat menaikkan AP'),
 ('E2','native1280 vs native640','Detail sumber dapat meningkatkan recall'),('E3','tile dari native1280','Objek lebih besar relatif terhadap input'),
 ('E4','SAHI','Recall dapat naik dengan biaya latensi dan FP'),('E5','head P2','Stride4 dapat membantu objek kecil'),
 ('E6','mosaic dimatikan','Uji dampak augmentasi satu faktor'),('E7','FP32 vs INT8','Uji biaya CPU dan perubahan AP')]
display(pd.DataFrame(plan,columns=['id','perlakuan','hipotesis']))
pd.DataFrame(plan,columns=['id','perlakuan','hipotesis']).to_csv(OUT/'rencana_eksperimen.csv',index=False)

## 6. E0 baseline dan E1 skala model

**Langkah:**

1. **E0:** latih `yolo26n.pt` (bobot pralatih) dengan `IMG_SIZE = 640` dan pengaturan bawaan.
2. Tampilkan kurva loss dan metrik dari `results.csv`. Apakah model sudah konvergen? Apakah ada tanda overfitting?
3. Evaluasi E0 pada split **val** dengan semua alat dari Bagian 4 (Ultralytics, COCOeval, recall per bin, latensi CPU), lalu `catat("E0", ...)`.
4. **E1:** ulangi langkah 1–3 dengan `yolo26s.pt`. Semua pengaturan lain sama dengan E0.
5. Setelah setiap eksperimen selesai, unduh folder `runs/<id>` dari tab *Output*, lalu unggah ke Kaggle Dataset privat kelompok (misalnya `fp-dm-runs-kelompok-XX`). Tambahkan dataset itu sebagai input, lalu set `LATIH[id] = False`.
6. Bila kuota GPU memungkinkan, ulangi E0 dengan satu seed lain untuk memperkirakan variasi antar-run. Selisih yang lebih kecil dari variasi ini **tidak boleh** diklaim sebagai perbaikan.

**Tulis:** perbandingan E0 vs E1 (akurasi, AP-small, latensi). Model mana yang menjadi titik awal untuk E2–E6, dan mengapa?

In [ ]:
if 'E0' in EVALUASI:
    if not BASE:raise ValueError('Input UTS belum tersedia')
    MODEL_PATHS['E0']=train_or_load('E0','yolo26n.pt',BASE_YAML,640,CONFIG)
    HASIL_VAL['E0'],_=evaluate_run('E0',MODEL_PATHS['E0'],BASE/'images/val',BASE_YAML,640,GT_VAL,CONFIG)

In [ ]:
if 'E1' in EVALUASI:
    if not BASE:raise ValueError('Input UTS belum tersedia')
    MODEL_PATHS['E1']=train_or_load('E1','yolo26s.pt',BASE_YAML,640,CONFIG)
    HASIL_VAL['E1'],_=evaluate_run('E1',MODEL_PATHS['E1'],BASE/'images/val',BASE_YAML,640,GT_VAL,CONFIG)

In [ ]:
for run in ['E0','E1']:
    curve=OUT/'runs'/run/'results.csv'
    if curve.exists():
        data=pd.read_csv(curve);data.columns=data.columns.str.strip()
        import matplotlib.pyplot as plt
        cols=[c for c in data if 'loss' in c or 'mAP' in c]
        data[cols].plot(subplots=True,figsize=(9,12));plt.tight_layout();plt.savefig(OUT/f'06_kurva_{run}.png');plt.show()

**Catatan kelompok.** E0 dan E1 M1 lokal tercatat masing-masing AP-small 0,0234 dan 0,0243. E1 M1 memakai YOLO26n freeze=10. E1 YOLO26s pada sel di atas masih memerlukan pelatihan atau bobot yang cocok. Kurva konvergensi dan ulangan seed perlu dilengkapi sebelum menyimpulkan perbedaan kecil antarmodel.

## 7. E2 resolusi tinggi

Dataset UTS sudah di-letterbox ke 640, sehingga detail objek kecil sudah hilang. Melatih dengan `imgsz=1280` pada dataset 640 **tidak** menambah informasi.

**Langkah:**

1. Jalankan ulang notebook UTS kelompok dengan `IMG_SIZE = 1280`, dengan `SEED`, `MANUAL_EXCLUDE`, dan `PILIH_VERSI` yang **sama** persis. Gunakan nama kelompok berakhiran `_1280` agar arsipnya terpisah.
2. Tambahkan arsip itu sebagai input dan salin ke `TEMP / "dataset1280"`, lalu tulis `data.yaml` dengan `path:` absolut.
3. Verifikasi bahwa daftar file per split **sama** dengan dataset 640 (bandingkan nama file). Bila berbeda, split tidak sebanding dan E2 tidak sah.
4. Ulangi profil ukuran (Bagian 3) untuk dataset 1280. Berapa persen kotak yang sekarang ≥ 16 px?
5. Latih E2 dengan model dari Bagian 6 dan `imgsz=1280`. Bila memori GPU tidak cukup, turunkan `batch` dan catat nilainya.
6. Evaluasi pada split val **dengan `imgsz=1280`**, catat hasil termasuk latensi CPU pada 1280.

**Tulis:** apakah kenaikan AP-small sepadan dengan kenaikan latensi?

In [ ]:
need_native=bool(set(EVALUASI)&{'E2','E2_control_native640','E3','E4'}) or EKSPOR_TILE or JALANKAN_TEST
if INPUT_NATIVE:
    if not BASE:raise ValueError('Audit UTS640 terlebih dahulu')
    NATIVE,NATIVE_YAML,GEO=native_data(INPUT_NATIVE,TEMP,BASE,OUT,KONFIRMASI_NATIVE)
elif need_native:
    raise ValueError('Isi INPUT_NATIVE dengan ekspor1280 dari citra asli atau paket native ketua')
else:
    print('E2–E4 menunggu input native1280 serta checkpoint yang sesuai.')

In [ ]:
if NATIVE:
    native_rows=[]
    for split in ['train','val']:
        for f in (NATIVE/'labels'/split).glob('*.txt'):
            native_rows += [{'split':split,'stem':f.stem,'sisi_px':min(r[3:])*1280} for r in read_labels(f)]
    df=pd.DataFrame(native_rows);df.to_csv(OUT/'profil_ukuran_1280.csv',index=False)
    display(df.assign(ge16=df.sisi_px>=16).groupby('split').ge16.mean().mul(100).rename('persen_ge16'))

In [ ]:
# Kontrol E2 dibangun dari ekspor640 pada paket pasangan native yang sama.
for run,size in [('E2_control_native640',640),('E2',1280)]:
    if run not in EVALUASI:continue
    if size==1280:root,yml=NATIVE,NATIVE_YAML
    else:
        candidate=NATIVE.parent/'640'
        if not (candidate/'images/train').exists():raise ValueError('Tambahkan ekspor native640 pasangan E2 untuk kontrol yang adil')
        root=TEMP/'kontrol_native640'
        if not root.exists():shutil.copytree(candidate,root)
        for split in ['train','val']:
            if {p.stem for p in (root/'images'/split).glob('*.jpg')} != {p.stem for p in (BASE/'images'/split).glob('*.jpg')}:raise ValueError('ID kontrol native640 berbeda')
        yml=write_yaml(root)
    MODEL_PATHS[run]=train_or_load(run,'yolo26n.pt',yml,size,CONFIG)
    HASIL_VAL[run],_=evaluate_run(run,MODEL_PATHS[run],root/'images/val',yml,size,GT_VAL,CONFIG,GEO)

**Catatan kelompok.** E2 lokal dengan pasangan 1.600 citra native train+val memberikan AP-small 0,021894→0,042689 dan median CPU 28,61→120,37 ms. Recall pada confidence0,05/IoU0,50 meningkat 0,279819→0,363265. Hasil ini berasal dari repo ketua dengan lima epoch dan batch4. Sel baru akan mencatat hasil eksekusi Kaggle beserta lingkungan dan threshold val-nya.

## 8. E3 pelatihan dengan tiling

Tiling memotong citra resolusi tinggi menjadi potongan berukuran 640, sehingga model belajar dari objek pada resolusi aslinya tanpa biaya memori `imgsz=1280`.

**Langkah:**

1. Dari `dataset1280`, potong **hanya split train** menjadi tile 640×640 dengan overlap (misalnya 20%). Split val dan test **tidak** dipotong.
2. Untuk setiap tile, hitung ulang koordinat kotak relatif terhadap tile. Kotak yang terpotong tepi tile: pertahankan bila sebagian besar luasnya (misalnya ≥ 50%) masih di dalam tile, lalu klip ke batas tile. Tulis ambang yang dipilih dan alasannya.
3. Tentukan nasib tile tanpa objek: simpan sebagian (misalnya maksimum 1 tile kosong per 3 tile berobjek) agar model tetap belajar latar belakang.
4. Visualisasikan 6 tile beserta kotaknya untuk memastikan koordinat benar.
5. Buat `data.yaml` baru: `train` menunjuk ke folder tile, `val` dan `test` tetap ke `dataset1280`.
6. Latih E3 dengan `imgsz=640`. Evaluasi pada val dengan **dua cara**: (a) `imgsz=1280` utuh, dan (b) SAHI (Bagian 9). Tulis di `catatan` cara mana yang dipakai.

In [ ]:
if EKSPOR_TILE or 'E3' in EVALUASI:
    if not NATIVE:raise ValueError('E3 membutuhkan native1280')
    TILE_YAML=export_tiles(NATIVE,TEMP,OUT,tile=640,overlap=.2,visibility=.5)

In [ ]:
if TILE_YAML:
    display(Image.open(OUT/'08_enam_tile.png'))
    print((OUT/'audit_tile_native.json').read_text())

In [ ]:
if 'E3' in EVALUASI:
    MODEL_PATHS['E3']=train_or_load('E3','yolo26n.pt',TILE_YAML,640,CONFIG)
    HASIL_VAL['E3'],_=evaluate_run('E3',MODEL_PATHS['E3'],NATIVE/'images/val',NATIVE_YAML,1280,GT_VAL,CONFIG,GEO)

In [ ]:
print('E3a dievaluasi pada gambar val utuh1280. Untuk E3b pilih SAHI_MODEL="E3", masukkan "E4" dalam EVALUASI.')

**Catatan kelompok.** E3 lokal yang tersedia memakai tile384 dari olahan640 dan ambang visibilitas25%, dengan anggaran620 optimizer steps. AP-small kontrol0,014290 dan tile0,003391. Sel baru menggunakan sumber1280, tile640, overlap20%, visibilitas≥50%, maksimum satu tile kosong per tiga tile berobjek. Hasil E3 native baru menunggu eksekusi. Jumlah langkah training dicatat dalam results.csv karena banyaknya tile mengubah jumlah batch per epoch.

## 9. E4 inferensi terpotong (SAHI)

SAHI tidak melatih ulang model; ia memotong citra uji menjadi beberapa irisan, mendeteksi di setiap irisan, lalu menggabungkan hasilnya.

**Langkah:**

1. Pilih model terbaik sejauh ini (berdasarkan val). Muat dengan `sahi.AutoDetectionModel.from_pretrained(model_type="ultralytics", model_path=..., confidence_threshold=0.001, device=...)`.
2. Jalankan `get_sliced_prediction` pada setiap gambar val **resolusi tinggi** dengan `slice_height = slice_width = 640` dan `overlap_*_ratio` sekitar 0,2. Coba juga opsi `perform_standard_pred=True` (gabungan prediksi utuh dan irisan).
3. Ubah hasilnya ke COCO JSON prediksi dengan **koordinat yang sama** dengan ground truth (perhatikan perbedaan skala 640 vs 1280), lalu evaluasi dengan COCOeval dan recall per bin (Bagian 4).
4. Ukur latensi CPU **per gambar utuh**, termasuk semua irisan.
5. Coba paling banyak 3 kombinasi ukuran irisan dan overlap pada val, lalu catat semuanya.

**Tulis:** berapa kenaikan recall objek kecil, berapa kenaikan FP, dan berapa kali lipat latensinya?

In [ ]:
if 'E4' in EVALUASI:
    if SAHI_MODEL not in MODEL_PATHS:MODEL_PATHS[SAHI_MODEL]=checkpoint_path(SAHI_MODEL,CHECKPOINTS,INPUT_ROOT)
    if len(SAHI_PILIH)>3:raise ValueError('Maksimal tiga konfigurasi SAHI pada val')
    if SAHI_MODEL not in HASIL_VAL:
        HASIL_VAL[SAHI_MODEL],_=evaluate_run(SAHI_MODEL,MODEL_PATHS[SAHI_MODEL],NATIVE/'images/val',NATIVE_YAML,1280,GT_VAL,CONFIG,GEO)

In [ ]:
if 'E4' in EVALUASI:
    for i in SAHI_PILIH:
        run=f'E4_{SAHI_MODEL}_config{i+1}'
        HASIL_VAL[run],_=evaluate_run(run,MODEL_PATHS[SAHI_MODEL],NATIVE/'images/val',NATIVE_YAML,1280,GT_VAL,CONFIG,GEO,SAHI_CONFIGS[i])

In [ ]:
if 'E4' in EVALUASI:
    comparison=pd.DataFrame([{'id':k,**v} for k,v in HASIL_VAL.items() if k==SAHI_MODEL or k.startswith('E4_')])
    comparison.to_csv(OUT/'09_perbandingan_sahi.csv',index=False)
    display(comparison.reindex(columns=['id','AP_small','R_fixed','recall_lt8','recall_8_16','FP','latensi_cpu_ms']))

**Catatan kelompok.** HANDOFF ketua melaporkan E4 native pada checkpoint E2 yang sama dengan AP-small0,04265→0,03899, recall tetap0,3628→0,3732, median CPU120,0→307,7ms. CSV rinci E4 belum dilacak di repo publik sehingga angka ini tetap berlabel ringkasan HANDOFF. Evaluasi Kaggle baru membandingkan gambar penuh dan SAHI dengan GT serta konfigurasi benchmark yang sama, maksimal tiga konfigurasi val.

## 10. E5 head P2

Head P2 menambah level deteksi dengan stride 4, sehingga sel grid lebih rapat untuk objek kecil, dengan biaya FLOPs dan memori lebih besar.

**Langkah:**

1. Temukan file konfigurasi model YOLO26 di dalam paket `ultralytics` (folder `ultralytics/cfg/models/`). Pelajari contoh konfigurasi `-p2` yang sudah ada untuk versi YOLO sebelumnya.
2. Salin YAML YOLO26 ke `WORK / "yolo26-p2.yaml"`. Tambahkan cabang upsample ke level P2 di neck, lalu sertakan keluaran P2 di layer `Detect`. Jelaskan perubahan per baris di sel markdown.
3. Bangun model dengan `YOLO("yolo26-p2.yaml")`, cetak `model.info()`, dan bandingkan parameter serta GFLOPs dengan model aslinya.
4. Pindahkan bobot pralatih yang cocok dengan `.load("yolo26n.pt")` (atau skala yang dipilih). Laporkan berapa layer yang berhasil dipindahkan.
5. Latih E5 dengan pengaturan yang sama dengan model pembandingnya, lalu evaluasi dan catat.

In [ ]:
P2_YAML=None
if 'E5' in EVALUASI:
    import ultralytics
    found=list((Path(ultralytics.__file__).parent/'cfg/models').rglob('yolo26-p2.yaml'))
    if len(found)!=1:raise FileNotFoundError('Konfigurasi yolo26-p2.yaml perlu disediakan untuk versi Ultralytics ini')
    P2_YAML=OUT/'yolo26n-p2.yaml';shutil.copy2(found[0],P2_YAML)
    p2=load_yolo(P2_YAML)
    if 4 not in p2.model.stride.tolist():raise ValueError('Head P2 stride4 tidak ditemukan')
    p2.info();print(P2_YAML.read_text())

In [ ]:
if P2_YAML:
    cfg=yaml.safe_load(P2_YAML.read_text())
    for i,layer in enumerate(cfg['backbone']+cfg['head']):print(i,layer)
    print('Upsample menghubungkan fitur resolusi tinggi. Detect memasukkan P2 stride4 bersama P3,P4,P5.')

In [ ]:
if 'E5' in EVALUASI:
    MODEL_PATHS['E5']=train_or_load('E5',P2_YAML,BASE_YAML,640,CONFIG,{'pretrained':'yolo26n.pt'})
    HASIL_VAL['E5'],_=evaluate_run('E5',MODEL_PATHS['E5'],BASE/'images/val',BASE_YAML,640,GT_VAL,CONFIG)

**Catatan kelompok.** Pilot P2 lokal selesai dengan AP-small0,017198 terhadap kontrol0,024085. Transfer layer dan parameter perlu dicatat dari keluaran training. Sel ini menyalin konfigurasi P2 paket Ultralytics serta memverifikasi stride4 sebelum dipakai.

## 11. E6 augmentasi

**Langkah:**

1. Pilih paling banyak **dua** perubahan augmentasi yang masuk akal untuk gejala FAW, misalnya `copy_paste`, rentang `scale`, `mosaic`/`close_mosaic`, `degrees`, atau `fliplr`/`flipud`. Tulis alasannya.
2. Tampilkan satu batch gambar latih hasil augmentasi (`train_batch*.jpg` di folder run) untuk memastikan kotak tidak rusak dan objek kecil tidak hilang.
3. Latih E6 dengan model pembanding yang sama, lalu evaluasi dan catat.

**Tulis:** augmentasi mana yang membantu, mana yang merugikan, dan mengapa?

In [ ]:
if 'E6' in EVALUASI:
    MODEL_PATHS['E6']=train_or_load('E6','yolo26n.pt',BASE_YAML,640,CONFIG,{'mosaic':0.0})
    HASIL_VAL['E6'],_=evaluate_run('E6',MODEL_PATHS['E6'],BASE/'images/val',BASE_YAML,640,GT_VAL,CONFIG)

In [ ]:
if 'E6' in EVALUASI:
    batches=sorted((OUT/'runs/E6').glob('train_batch*.jpg'))
    if batches:display(Image.open(batches[0]))

**Catatan kelompok.** Satu faktor diubah pada E6 yaitu mosaic menjadi0. Pilot lokal memberikan AP-small0,023398 terhadap kontrol0,024085. Uji lima epoch masih memerlukan pemeriksaan kurva dan variasi seed.

## 12. Pemilihan model final dan uji akhir pada test

**Langkah:**

1. Tampilkan `eksperimen.csv` (hasil val) yang diurutkan berdasarkan metrik utama yang sudah kelompok tetapkan **sebelum** melihat hasil. Contohnya `AP_small`, dengan `mAP50_95` sebagai pemecah seri.
2. Pilih paling banyak **dua** model final: satu yang paling akurat, dan satu yang paling seimbang antara akurasi dan latensi CPU. Tulis alasannya.
3. Jalankan evaluasi lengkap (Bagian 4) pada split **test**, **satu kali**, untuk model-model final. Gunakan ambang confidence operasional yang dipilih dari val, bukan dari test.
4. Simpan hasilnya sebagai baris terpisah di `eksperimen.csv` (misalnya `E5_test`) dan simpan gambar kurva PR, F1, serta confusion matrix.
5. Bandingkan angka val dan test. Selisih besar menandakan apa?

In [ ]:
if HASIL_VAL:
    display(pd.DataFrame([{'id':k,**v} for k,v in HASIL_VAL.items()]).sort_values('AP_small',ascending=False))
FINAL_CONFIG=None;TEST_ROWS=None
if JALANKAN_TEST:
    if MODEL_FINAL not in HASIL_VAL:raise ValueError('Evaluasi val kandidat final pada eksekusi ini sebelum mengunci test')
    FINAL_PATH=MODEL_PATHS[MODEL_FINAL]
    FINAL_CONFIG={'id':MODEL_FINAL,'sha256':sha256(FINAL_PATH),'imgsz':HASIL_VAL[MODEL_FINAL]['imgsz'],
                  'conf_operasional':HASIL_VAL[MODEL_FINAL]['conf_operasional'],'seed':SEED}
    dump(OUT/'konfigurasi_final_terkunci.json',FINAL_CONFIG)

In [ ]:
if JALANKAN_TEST:
    marker=OUT/'uji_test_selesai.json'
    if marker.exists():
        print('Hasil test sudah ada',json.loads(marker.read_text()))
        TEST_ROWS=pd.read_csv(OUT/(MODEL_FINAL+'_test')/'TP_FP_FN.csv').to_dict('records')
        for r in TEST_ROWS:
            if isinstance(r['bbox'],str):r['bbox']=json.loads(r['bbox'])
    else:
        size=FINAL_CONFIG['imgsz'];test_root=NATIVE if size==1280 else BASE
        if not (test_root/'images/test').is_dir():raise ValueError('Sumber native test178 belum disiapkan. Builder E2 lokal hanya train+val.')
        if {p.stem for p in (test_root/'images/test').glob('*.jpg')} != {p.stem for p in (BASE/'images/test').glob('*.jpg')}:raise ValueError('ID test native berbeda')
        gt_test=coco_gt(RECORDS,'test',OUT)
        TEST_METRICS,TEST_ROWS=evaluate_run(MODEL_FINAL+'_test',FINAL_PATH,test_root/'images/test',write_yaml(test_root),size,gt_test,CONFIG,GEO,split='test',fixed_conf=FINAL_CONFIG['conf_operasional'])
        dump(marker,{'config':FINAL_CONFIG,'metrics':TEST_METRICS})

**Catatan kelompok.** Kandidat sementara E2 native1280 dipilih dari validation. Hasil test178 belum tersedia pada arsip publik M2. Kunci checkpoint dan confidence dari val sebelum mengaktifkan JALANKAN_TEST. Perbandingan val/test diisi setelah pengujian yang tercatat.

## 13. Analisis error

**Langkah:**

1. Dengan daftar FN dan FP dari Bagian 4.3 (model final, split test), buat montase: 12 FN terkecil, 12 FP dengan confidence tertinggi, dan 6 TP sebagai pembanding.
2. Beri label penyebab untuk setiap contoh di montase (isi tabel manual di sel markdown): `objek kecil`, `blur`, `terhalang`, `mirip gejala lain` (bercak penyakit, kerusakan mekanis), `bayangan/pantulan`, `label salah/terlewat`, `lokalisasi meleset`.
3. Gabungkan hasil per gambar dengan `inventaris.csv` dari UTS (kunci: nama file/`stem`) untuk mengambil `blur_score`, `terang`, `pengunggah`, dan `source_id`.
4. Hitung recall per **kuartil blur**, per **pengunggah**, dan per **sumber video**. Tampilkan sebagai grafik batang.
5. Untuk gambar yang punya **label acuan kelompok** (UTS Bagian 8), bandingkan kinerja model terhadap label kelas dan terhadap label acuan. Apakah sebagian "error" sebenarnya kesalahan label?
6. Simpan montase sebagai `13_fn.png`, `13_fp.png`, `13_tp.png` dan tabel sebagai `analisis_error.csv`.

In [ ]:
if TEST_ROWS is not None:error_analysis(TEST_ROWS,BASE,UTS_ROOT,OUT)

In [ ]:
if TEST_ROWS is not None:
    for f in sorted(OUT.glob('13_*.png')):display(Image.open(f))

In [ ]:
if TEST_ROWS is not None:
    display(pd.read_csv(OUT/'penyebab_error_isi_manual.csv').head(12))
    print('Isi penyebab setelah inspeksi visual, lalu hitung jumlah per penyebab dari tabel tersebut.')

In [ ]:
# Label acuan UTS asli memiliki koordinat citra asal. Untuk menilai model pada ekspor640,
# gunakan hanya acuan yang sudah ditransformasi ke640 dan memiliki ID evaluasi yang cocok.
if TEST_ROWS is not None:
    if LABEL_KELAS_640 and LABEL_ACUAN_640:
        path=OUT/(MODEL_FINAL+'_test')/'coco_test_prediksi.json'
        compare_reference_labels(coco_gt(RECORDS,'test',OUT),json.loads(path.read_text()),
             LABEL_KELAS_640,LABEL_ACUAN_640,FINAL_CONFIG['conf_operasional'],OUT)
    else:print('Perbandingan acuan menunggu label acuan yang sudah ditransformasi ke640.')

**Penyebab error.** Tabel penyebab_error_isi_manual.csv disiapkan dari contoh TP/FP/FN sesudah evaluasi. Pemeriksa mengisi penyebab seperti blur, oklusi, objek kecil, label terlewat, atau lokalisasi. Frekuensi penyebab menunggu inspeksi contoh.

## 14. E7 ekspor INT8 dan latensi CPU

**Langkah:**

1. Ekspor model final ke format CPU: ONNX (FP32) dan satu format INT8, misalnya OpenVINO atau TFLite dengan `int8=True`. Kalibrasi INT8 memakai `data=` yang menunjuk ke **train/val**, bukan test.
2. Evaluasi setiap model hasil ekspor pada split test dengan `YOLO("<file ekspor>").val(...)` serta COCOeval, lalu catat penurunan mAP dan AP-small dibanding PyTorch.
3. Ukur latensi CPU semua format dengan fungsi Bagian 4.4 pada mesin yang sama.
4. Catat ukuran file setiap format (MB).
5. Buat grafik **akurasi vs latensi** (sumbu x: latensi CPU ms, sumbu y: AP_small atau mAP50-95) untuk semua eksperimen dan format.

**Tulis:** format mana yang paling layak untuk perangkat lapangan (ponsel atau komputer mini di kebun)?

In [ ]:
if JALANKAN_INT8:
    if FINAL_CONFIG is None:raise ValueError('Kunci kandidat dan jalankan evaluasi val terlebih dahulu')
    subprocess.check_call([sys.executable,'-m','pip','install','-q','openvino','nncf','onnx'])
    source_root=NATIVE if FINAL_CONFIG['imgsz']==1280 else BASE
    calibration=sorted((source_root/'images/train').glob('*.jpg'))
    chosen=random.Random(SEED).sample(calibration,min(320,len(calibration)))
    manifest=OUT/'kalibrasi_train.txt';manifest.write_text(''.join(str(p)+'\n' for p in chosen))
    cal=OUT/'kalibrasi.yaml';cal.write_text(yaml.safe_dump({'path':str(OUT),'train':str(manifest),'val':str(source_root/'images/val'),'names':CLASS_NAMES}))
    size=FINAL_CONFIG['imgsz']
    for fmt,extra in [('onnx',{}),('openvino',{'int8':True,'data':str(cal),'split':'train'})]:
        p=load_yolo(FINAL_PATH).export(format=fmt,imgsz=size,batch=1,device='cpu',**extra)
        EXPORTS.append(p)

In [ ]:
if JALANKAN_INT8:
    for i,p in enumerate(EXPORTS):
        size=FINAL_CONFIG['imgsz'];r=NATIVE if size==1280 else BASE
        run=f'E7_{i+1}_test'
        marker=OUT/run/'metrik.json'
        if marker.exists():print('Hasil ekspor tersimpan',marker);continue
        evaluate_run(run,p,r/'images/test',write_yaml(r),size,coco_gt(RECORDS,'test',OUT),CONFIG,GEO,split='test',fixed_conf=FINAL_CONFIG['conf_operasional'])

In [ ]:
if (OUT/'eksperimen.csv').exists():
    import matplotlib.pyplot as plt
    data=pd.read_csv(OUT/'eksperimen.csv')
    if 'latensi_cpu_ms' in data:
        measured=data.dropna(subset=['latensi_cpu_ms','AP_small'])
        if len(measured):
            ax=measured.plot.scatter(x='latensi_cpu_ms',y='AP_small')
            for _,r in measured.iterrows():ax.annotate(r.id,(r.latensi_cpu_ms,r.AP_small))
            plt.savefig(OUT/'14_akurasi_latensi.png',dpi=140);plt.show()

**Catatan kelompok.** Pilot E7 kontrol640 tercatat selesai pada HANDOFF ketua. Perbandingan ekspor untuk kandidat final1280 menunggu eksekusi. Kalibrasi baru memakai train320 sampel dan evaluasi memakai threshold val yang terkunci. Pilihan format lapangan menunggu AP, latensi, dan ukuran ekspor pada mesin yang sama.

## 15. Ringkasan hasil dan pengumpulan

**Langkah:**

1. Tampilkan `eksperimen.csv` final dan grafik selisih setiap eksperimen terhadap E0 (AP-small, recall < 16 px, mAP50-95, latensi).
2. Kemas ke `/kaggle/working/<KELOMPOK>_hasil.zip`: `eksperimen.csv`, semua CSV analisis, semua gambar `*.png`, `best.pt` model final, file ekspor INT8, dan YAML kustom (bila ada).
3. Pastikan semua saklar `LATIH` bernilai `False`, lalu **Save Version → Save & Run All**. Notebook harus berjalan tuntas tanpa melatih ulang.
4. Bagikan notebook ke dosen dan unggah laporan serta slide presentasi sesuai instruksi kelas.

**Luaran yang dikumpulkan:** notebook Kaggle (bisa dijalankan ulang), `<KELOMPOK>_hasil.zip`, laporan berformat artikel ilmiah, dan presentasi 10 menit dengan demo inferensi pada foto baru.

In [ ]:
if (OUT/'eksperimen.csv').exists():
    display(pd.read_csv(OUT/'eksperimen.csv'))
else:
    # Header disiapkan tanpa menambahkan baris eksperimen yang belum dieksekusi.
    cols=['id','perlakuan','model','imgsz','epochs','seed','tanggal','P','R','F1','conf_operasional','mAP50','mAP50_95',
          'AP_small','AP_medium','AR100','recall_lt8','recall_8_16','recall_16_32','recall_32_64','recall_ge64',
          'latensi_cpu_ms','params_M','gflops','ukuran_MB','status']
    pd.DataFrame(columns=cols).to_csv(OUT/'eksperimen.csv',index=False)
status={'scope':'Progress M2','data_ready':BASE is not None,'native_ready':NATIVE is not None,
        'runs_dihitung':list(HASIL_VAL),'history_repo_commit':'e376ec2','test_dijalankan':JALANKAN_TEST,
        'latih':LATIH,'publication':'Nomor versi Kaggle dicatat setelah Save & Run All berhasil'}
dump(OUT/'status_eksekusi.json',status)
print(json.dumps(status,indent=2))

In [ ]:
arsip=package(OUT,KELOMPOK,FINAL_PATH,EXPORTS)
print('Arsip progres',arsip)
try:
    from IPython.display import FileLink
    display(FileLink(str(arsip)))
except ImportError:
    print('Arsip tersedia pada path di atas.')

## Pertanyaan yang wajib dijawab di laporan

1. Bagaimana distribusi ukuran objek FAW di dataset (Bagian 3)? Berapa persen kotak < 16 px, dan apa artinya bagi pilihan eksperimen?
2. Seberapa baik baseline YOLO26 (E0, E1)? Di bin ukuran mana recall paling rendah?
3. Teknik objek kecil mana (E2–E6) yang paling menaikkan AP-small dan recall objek kecil? Apakah kenaikannya lebih besar daripada variasi antar-seed?
4. Berapa biaya latensi CPU setiap teknik? Model dan format mana yang Anda rekomendasikan untuk perangkat lapangan, dan mengapa?
5. Apa penyebab error yang paling sering (Bagian 13)? Berapa bagian error yang sebenarnya berasal dari masalah label atau kualitas citra yang sudah terdeteksi di UTS?
6. Bandingkan hasil val dan test. Apakah ada tanda overfitting terhadap val?
7. Jika punya satu bulan lagi, perbaikan data atau model apa yang akan Anda lakukan lebih dulu? Dukung dengan bukti dari notebook ini.

### Jawaban sementara kelompok pada M2
1. Audit ulang menemukan4.952 dari13.904 kotak (35,62%) bersisi<16px pada acuan640. Train36,68%, val33,79%, test29,42%. Resolusi native dan tiling menjadi prioritas.
2. E0 lokal AP-small0,023398 dan E1 freeze10 sebesar0,024281. E1 YOLO26s dan recall per bin masih perlu eksekusi.
3. Pasangan E2 native menghasilkan AP-small0,021894→0,042689. Variasi antar-seed belum tersedia. Pilot E3 berbahan640 belum mengungguli kontrolnya.
4. E2 medianCPU28,61→120,37ms. E4 medianCPU120,0→307,7ms dalam protokolnya sendiri. Kandidat lapangan menunggu pengukuran ekspor final.
5. Frekuensi penyebab error masih menunggu inspeksi TP/FP/FN. Dugaan label/kualitas perlu didukung contoh dan tabel.
6. Evaluasi test178 belum dijalankan. Perbandingan val/test masih menunggu kandidat terkunci.
7. Prioritas lanjutan ialah pemeriksaan label kecil, pengumpulan citra beragam dan background, serta ulangan seed. Dasarnya35,62% kotak<16px dan skor pilot yang masih rendah.
